# Faculty-Course Matching: AY 2026-2027 (Fall + Spring)

Solves the faculty-to-course teaching assignment problem for both Fall 2026 and Spring 2027 as a **single** min-cost max-flow LP.

## Graph Architecture
```
BOS -> Faculty_i -> FallGW_i  (cap=U_fall)  -> Fall Courses  -> Fall Completions  -> EOS
                 -> SpringGW_i (cap=U_spring) -> Spring Courses -> Spring Completions -> EOS
```
Gateway nodes enforce per-semester teaching limits. BOS->Faculty capacity = yearly total.

In [ ]:
include(joinpath(@__DIR__, "Include.jl"));

## Generate the graph edgelist and load data

In [ ]:
# Generate the edgelist programmatically from CSV inputs
graph_info = generate_edgelist(
    faculty_csv = joinpath(_PATH_TO_DATA, "Faculty.csv"),
    fall_courses_csv = joinpath(_PATH_TO_DATA, "Courses-Fall-2026.csv"),
    spring_courses_csv = joinpath(_PATH_TO_DATA, "Courses-Spring-2027.csv"),
    fall_preferences_csv = joinpath(_PATH_TO_DATA, "Faculty-Course-Preferences-Fall-2026.csv"),
    spring_preferences_csv = joinpath(_PATH_TO_DATA, "Faculty-Course-Preferences-Spring-2027.csv"),
    output_edgelist = joinpath(_PATH_TO_DATA, "Faculty-Courses-Bipartite-AY-2026-2027.edgelist")
);

In [ ]:
# Unpack graph metadata
faculty_df = graph_info["faculty_df"];
fall_courses_df = graph_info["fall_courses_df"];
spring_courses_df = graph_info["spring_courses_df"];

println("Nodes: $(graph_info["N_nodes"]) | Faculty: $(graph_info["N_faculty"]) | Fall courses: $(graph_info["N_fall_courses"]) | Spring courses: $(graph_info["N_spring_courses"])")
println("Total flow F = $(graph_info["F"])")

## Parse the edgelist and build the directed graph model

In [ ]:
"""
Parse a single edge record from the edgelist file.
Format: source,target,cost,lb,ub
"""
function edgerecordparser(record::String, delim::Char=',')
    fields = split(record, delim)
    if length(fields) < 5
        return nothing
    end
    source = parse(Int, fields[1])
    target = parse(Int, fields[2])
    cost = parse(Float64, fields[3])
    l = parse(Float64, fields[4])
    u = parse(Float64, fields[5])
    return (source, target, cost, l, u)
end;

In [ ]:
path_to_edge_file = joinpath(_PATH_TO_DATA, "Faculty-Courses-Bipartite-AY-2026-2027.edgelist");
myedgemodels = MyConstrainedGraphEdgeModels(path_to_edge_file, edgerecordparser, delim=',', comment='#');

In [ ]:
directedgraphmodel = let
    s = graph_info["BOS"]
    t = graph_info["EOS"]
    build(MyDirectedBipartiteGraphModel, (
        s = s,
        t = t,
        edges = myedgemodels
    ))
end;

## Set up capacity bounds

In [ ]:
bounds = let
    capacity = directedgraphmodel.capacity
    number_of_edges = length(directedgraphmodel.edges)
    bounds = Array{Float64,2}(undef, number_of_edges, 2)

    for (k, v) in directedgraphmodel.edgesinverse
        edge_index = k
        lb = capacity[v][1]
        ub = capacity[v][2]
        bounds[edge_index, 1] = lb
        bounds[edge_index, 2] = ub
    end

    bounds
end;

### Optional: Runtime capacity overrides
All per-semester capacities are already encoded in `Faculty.csv` and baked into the edgelist.
Use the cell below only for ad-hoc experiments (e.g., "what if Harimoto comes back for Fall?").

In [ ]:
# Example: override a specific faculty's gateway capacity at runtime
# Uncomment and modify as needed:
#
# let
#     idx = findfirst(==("Harimoto"), faculty_df[!, :name])
#     update_edge_capacity!(directedgraphmodel, bounds,
#         source = graph_info["faculty_nodes"][idx],
#         target = graph_info["fall_gateway_nodes"][idx],
#         lb = 0.0, ub = 1.0  # enable fall teaching
#     )
# end

## Set up cost (objective) vector

In [ ]:
c = let
    number_of_edges = length(directedgraphmodel.edges)
    weight_array = Array{Float64,1}(undef, number_of_edges)
    weights = directedgraphmodel.edges

    for (k, v) in directedgraphmodel.edgesinverse
        edge_index = k
        weight_array[edge_index] = weights[v]
    end

    weight_array
end;

### Manual cost overrides
Set cost = -1.0 for specific faculty-course pairs where we have strong preferences
beyond the survey data. In the gateway architecture, we target (gateway_node, course_node) edges.

In [ ]:
c = let
    weight_array = copy(c)

    # Helper: look up gateway and course nodes for a (faculty, course, semester) triple
    function set_override!(wa, faculty_name, course_name, semester; wv=-1.0)
        idx = findfirst(==(faculty_name), faculty_df[!, :name])
        if isnothing(idx)
            @warn "Faculty not found: $faculty_name"
            return
        end

        if semester == :fall
            gw = graph_info["fall_gateway_nodes"][idx]
            ci = findfirst(==(course_name), fall_courses_df[!, :course])
            if isnothing(ci)
                @warn "Fall course not found: $course_name"
                return
            end
            cn = graph_info["fall_course_nodes"][ci]
        else
            gw = graph_info["spring_gateway_nodes"][idx]
            ci = findfirst(==(course_name), spring_courses_df[!, :course])
            if isnothing(ci)
                @warn "Spring course not found: $course_name"
                return
            end
            cn = graph_info["spring_course_nodes"][ci]
        end

        update_cost_array!(directedgraphmodel, wa, wv=Float64(wv), faculty=gw, course=cn)
    end

    # --- FALL: Core undergraduate ---
    set_override!(weight_array, "Godwin",   "ENGRI-1120", :fall)
    set_override!(weight_array, "Celik",    "CHEME-2880", :fall)
    set_override!(weight_array, "Duncan",   "ENGRD-2190", :fall)
    set_override!(weight_array, "Hanrath",  "CHEME-3130", :fall)
    set_override!(weight_array, "Goldfarb", "CHEME-3240", :fall)
    set_override!(weight_array, "Bauer",    "CHEME-4320", :fall)

    # --- FALL: Elective undergraduate ---
    set_override!(weight_array, "Varner", "CHEME-4800", :fall)
    set_override!(weight_array, "Varner", "CHEME-5660", :fall)
    set_override!(weight_array, "Tester", "CHEME-4840", :fall)
    set_override!(weight_array, "Tester", "CHEME-4880", :fall)

    # --- FALL: M.Eng ---
    set_override!(weight_array, "Bauer",  "CHEME-5020", :fall)
    set_override!(weight_array, "Bauer",  "CHEME-5650", :fall)
    set_override!(weight_array, "Cleary", "CHEME-5770", :fall)

    # --- FALL: Graduate core ---
    set_override!(weight_array, "Escobedo", "CHEME-6110", :fall)
    set_override!(weight_array, "Yue",      "CHEME-6130", :fall)
    set_override!(weight_array, "Stroock",   "CHEME-6230", :fall)
    set_override!(weight_array, "Kowal",    "CHEME-6920", :fall)

    # --- FALL: Graduate elective ---
    set_override!(weight_array, "Kalra",   "CHEME-5310", :fall)
    set_override!(weight_array, "Putnam",  "CHEME-6310", :fall)
    set_override!(weight_array, "Koch",    "CHEME-6440", :fall)
    set_override!(weight_array, "Hanrath", "CHEME-6662", :fall)
    set_override!(weight_array, "Tester",  "CHEME-6681", :fall)
    set_override!(weight_array, "Tester",  "CHEME-6660", :fall)
    set_override!(weight_array, "You",     "CHEME-6800", :fall)
    set_override!(weight_array, "You",     "CHEME-6810", :fall)
    set_override!(weight_array, "You",     "CHEME-6830", :fall)
    set_override!(weight_array, "You",     "CHEME-6840", :fall)

    # --- SPRING: Add overrides here as needed ---
    # set_override!(weight_array, "FacultyName", "CHEME-XXXX", :spring)

    weight_array
end;

## Build and solve the LP

In [ ]:
# Flow conservation: net flow at each node
b = let
    number_of_nodes = length(directedgraphmodel.nodes)
    b = zeros(number_of_nodes)
    s = directedgraphmodel.source
    t = directedgraphmodel.sink
    F = graph_info["F"]

    b[s] = -F  # source supplies flow
    b[t] = F   # sink absorbs flow

    b
end;

In [ ]:
# Incidence matrix: node-edge flow conservation constraints
A = let
    number_of_nodes = length(directedgraphmodel.nodes)
    number_of_edges = length(directedgraphmodel.edges)
    A = zeros(number_of_nodes, number_of_edges)

    for (k, v) in directedgraphmodel.edgesinverse
        edge_index = k
        s = v[1]
        t = v[2]
        A[s, edge_index] = -1.0  # outgoing
        A[t, edge_index] = 1.0   # incoming
    end

    A
end;

In [ ]:
primal_problem = build(MyLinearProgrammingProblemModel, (
    c = -c,
    A = A,
    b = b,
    lb = bounds[:, 1],
    ub = bounds[:, 2]
));

In [ ]:
primal_solution_dictionary = let
    primal_solution = nothing
    try
        primal_solution = solve(primal_problem)
    catch error
        println(error)
    end
    primal_solution
end;

## Extract flow and assignments

In [ ]:
flow = let
    flow_dictionary = Dict{Tuple{Int,Int}, Float64}()
    primal_flow_vector = primal_solution_dictionary["argmax"]

    for (k, v) in directedgraphmodel.edgesinverse
        edge_index = k
        s = v[1]
        t = v[2]
        flow_dictionary[(s, t)] = primal_flow_vector[edge_index]
    end

    flow_dictionary
end;

In [ ]:
# Extract Fall assignments
fall_matching = extract_matching(flow,
    graph_info["fall_gateway_nodes"],
    graph_info["fall_course_nodes"],
    faculty_df, fall_courses_df
);

# Extract Spring assignments
spring_matching = extract_matching(flow,
    graph_info["spring_gateway_nodes"],
    graph_info["spring_course_nodes"],
    faculty_df, spring_courses_df
);

## Fall 2026 Assignments

In [ ]:
let
    df = DataFrame()
    for (faculty_name, assigned_courses) in sort(collect(fall_matching), by=x->x[1])
        courses_str = isempty(assigned_courses) ? "<No Assignment>" : join(assigned_courses, "; ")
        title = ""
        if !isempty(assigned_courses)
            title = String(fall_courses_df[fall_courses_df.course .== assigned_courses[1], :title][1])
        end
        push!(df, (Faculty = faculty_name, Assigned_Courses = courses_str, Course_Title = title))
    end

    pretty_table(df;
        backend = :text,
        alignment = [:l, :l, :l],
        title = "Fall 2026 Teaching Assignments"
    )
end

## Spring 2027 Assignments

In [ ]:
let
    df = DataFrame()
    for (faculty_name, assigned_courses) in sort(collect(spring_matching), by=x->x[1])
        courses_str = isempty(assigned_courses) ? "<No Assignment>" : join(assigned_courses, "; ")
        title = ""
        if !isempty(assigned_courses)
            title = String(spring_courses_df[spring_courses_df.course .== assigned_courses[1], :title][1])
        end
        push!(df, (Faculty = faculty_name, Assigned_Courses = courses_str, Course_Title = title))
    end

    pretty_table(df;
        backend = :text,
        alignment = [:l, :l, :l],
        title = "Spring 2027 Teaching Assignments"
    )
end

## Summary: Faculty load across both semesters

In [ ]:
let
    df = DataFrame()
    for i in 1:graph_info["N_faculty"]
        name = String(faculty_df[i, :name])
        fall_courses = get(fall_matching, name, String[])
        spring_courses = get(spring_matching, name, String[])
        n_fall = length(fall_courses)
        n_spring = length(spring_courses)
        cap_fall = faculty_df[i, :U_fall]
        cap_spring = faculty_df[i, :U_spring]

        push!(df, (
            Faculty = name,
            Fall_Assigned = n_fall,
            Fall_Cap = cap_fall,
            Spring_Assigned = n_spring,
            Spring_Cap = cap_spring,
            Total = n_fall + n_spring,
            Yearly_Cap = cap_fall + cap_spring
        ))
    end

    pretty_table(df;
        backend = :text,
        alignment = [:l, :c, :c, :c, :c, :c, :c],
        title = "Faculty Load Summary AY 2026-2027"
    )
end

## Save results

In [ ]:
let
    path_to_output = joinpath(_PATH_TO_DATA, "Faculty-Course-Assignments-AY-2026-2027.jld2")

    save(path_to_output, Dict(
        "fall_matching" => fall_matching,
        "spring_matching" => spring_matching,
        "faculty" => faculty_df,
        "fall_courses" => fall_courses_df,
        "spring_courses" => spring_courses_df,
        "flow" => flow,
        "primal_solution_dictionary" => primal_solution_dictionary,
        "graph_info" => graph_info,
        "c" => c,
        "bounds" => bounds,
        "A" => A,
        "b" => b,
        "directedgraphmodel" => directedgraphmodel
    ))

    println("Results saved to: $path_to_output")
end